# EDA — Modèle de résiliation tarifaire (élasticité prix)

Notebook d'analyse exploratoire **orienté élasticité**, conçu pour un modèle de
résiliation qui alimentera un **optimiseur tarifaire**. L'objectif n'est pas
seulement de décrire les données, mais de **vérifier que la relation
prix → résiliation est identifiable, monotone et non biaisée** avant de modéliser.

**Ordre de lecture (du plus rentable au plus accessoire) :**
1. Setup & configuration des colonnes
2. Vue d'ensemble & qualité des données
3. Statistiques univariées (toutes les mesures)
4. **Cohérence des features prix** (sanity mécanique)
5. **Cœur élasticité : courbes prix → résiliation** (le test de viabilité)
6. **Confounding prix ↔ risque** (le piège n°1)
7. Corrélations, redondances (VIF), associations
8. Stabilité temporelle (PSI)
9. Scan de fuite (leakage)
10. Synthèse automatique

> Pour utiliser sur vos données : section 1, mettez `USE_SYNTHETIC = False`,
> chargez votre base, et **adaptez le dictionnaire `CFG`** à vos noms de colonnes.

## 1. Setup & configuration

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (9, 4.5)
plt.rcParams["axes.titlesize"] = 12
RNG = np.random.default_rng(42)

In [ ]:
# === CONFIGURATION : adaptez ces noms à VOTRE base ========================
USE_SYNTHETIC = True   # -> False pour charger vos données réelles

TARGET   = "resiliation"          # cible binaire : 1 = résilié, 0 = conservé
YEAR_COL = "annee_observation"    # millésime (stabilité temporelle) ; None si absent

CFG = {
    # --- bloc PRIX (cœur élasticité) ---------------------------------------
    "p_t":          "prm_propose_t",       # prix proposé = LEVIER de l'optimiseur (dynamique)
    "p_t1":         "prm_ref_t1",          # prix année -1 = ancre (figé)
    "p_t2":         "prm_t2",
    "p_t3":         "prm_t3",
    "delta_pct_t":  "delta_prm_pct_t",     # majoration % courante (DRIVER n°1, dynamique)
    "delta_eur_t":  "delta_prm_eur_t",     # majoration € courante (dynamique)
    "delta_pct_t1": "delta_prm_pct_t1",    # majoration % an dernier (figé)
    "delta_pct_t2": "delta_prm_pct_t2",    # majoration % il y a 2 ans (figé)
    "cumul_3a":     "hausse_cumul_pct_3a", # P_t/P_t3 - 1 (dynamique)
    "franchit_100": "is_franchit_centaine_t",
}

# Variables qui MODULENT l'élasticité (qui réagit comment)
MODULATORS_NUM = ["anciennete", "age_assure", "prix_vehicule", "age_vehicule",
                  "nb_enfants_en_charge", "multi_detention"]
MODULATORS_CAT = ["canal", "mode_paiement"]

# CONFONDEURS de risque : à inclure pour DÉ-BIAISER (jamais en ratio avec le prix)
CONFOUNDERS_NUM = ["bonus_malus", "sinistralite_3a", "nb_jeunes_conducteurs"]
CONFOUNDERS_CAT = ["classe_vehicule", "zone_geo"]
RISK_STRATIFY   = "bonus_malus"   # variable de risque pour la courbe "à risque constant"

# Comportemental (si disponible)
BEHAVIOR = ["nb_contacts_sc_12m", "nb_reclamations", "menace_resil_passee"]

FLAGS = ["flag_hist_incomplet"]
# =========================================================================

In [ ]:
def make_synthetic(n=25000, seed=42):
    '''Base synthetique : elasticite vraie + confounding prix/risque +
    modulation par segment + effet de seuil + new business sans historique.'''
    rng = np.random.default_rng(seed)
    anciennete   = np.clip(rng.exponential(6, n).round(), 0, 30).astype(int)
    age_assure   = np.clip(rng.normal(45, 14, n), 18, 90).round().astype(int)
    canal        = rng.choice(["web", "agent", "courtier"], n, p=[0.35, 0.45, 0.20])
    mode_paiement= rng.choice(["mensuel", "annuel"], n, p=[0.6, 0.4])
    multi_det    = rng.choice([1, 2, 3, 4], n, p=[0.55, 0.30, 0.10, 0.05])
    classe       = rng.choice(list("ABCDEFGH"), n)
    zone         = rng.choice([f"Z{i:02d}" for i in range(1, 21)], n)
    age_veh      = np.clip(rng.exponential(6, n).round(), 0, 25).astype(int)
    nb_enf       = rng.choice([0, 1, 2, 3, 4], n, p=[0.45, 0.25, 0.20, 0.07, 0.03])
    nb_jeunes    = rng.choice([0, 1, 2], n, p=[0.82, 0.15, 0.03])
    bonus_malus  = np.clip(rng.normal(0.85, 0.25, n), 0.5, 3.5).round(2)
    sinistres    = rng.poisson(np.clip(0.4 + (bonus_malus - 0.85), 0, None)).astype(int)
    cnum         = pd.Series(classe).map({c: i for i, c in enumerate("ABCDEFGH")}).values
    prix_veh     = (8000 + cnum * 4000 + rng.normal(0, 3000, n)).clip(2000).round(0)

    # Prime de référence CORRÉLÉE au risque -> confounding sur le NIVEAU
    p_t1 = (250 + 180 * bonus_malus + 25 * cnum + 220 * nb_jeunes
            + 0.004 * prix_veh + rng.normal(0, 60, n)).clip(120).round(2)

    # Majoration courante : revalorisation book (~+5%) + idiosyncratique
    delta_t  = rng.normal(0.05, 0.06, n).clip(-0.15, 0.60)
    delta_t1 = rng.normal(0.04, 0.05, n).clip(-0.10, 0.40)
    delta_t2 = rng.normal(0.03, 0.05, n).clip(-0.10, 0.40)

    p_t   = (p_t1 * (1 + delta_t)).round(2)
    d_eur = (p_t - p_t1).round(2)
    p_t2  = (p_t1 / (1 + delta_t1)).round(2)
    p_t3  = (p_t2 / (1 + delta_t2)).round(2)
    cumul = (p_t / p_t3 - 1)
    franch= (np.floor(p_t / 100) > np.floor(p_t1 / 100)).astype(int)

    # New business : pas d'historique complet
    incomplete = anciennete < 2
    p_t2  = p_t2.astype(float);  p_t3 = p_t3.astype(float)
    delta_t2 = delta_t2.astype(float); cumul = cumul.astype(float)
    p_t3[incomplete] = np.nan
    p_t2[anciennete < 1] = np.nan
    delta_t2[incomplete] = np.nan
    cumul[incomplete] = np.nan
    flag_inc = incomplete.astype(int)

    nb_contacts = rng.poisson(0.5, n)
    nb_reclam   = rng.poisson(0.1, n)
    menace      = (rng.random(n) < 0.05).astype(int)
    annee       = rng.choice([2023, 2024, 2025], n, p=[0.30, 0.35, 0.35])

    # Quelques manquants pour exercer le rapport qualité
    age_veh = age_veh.astype(float)
    age_veh[rng.random(n) < 0.03] = np.nan

    # Churn : élasticité vraie + risque (confounding) + modulation + seuil
    canal_eff = pd.Series(canal).map({"web": 0.5, "agent": -0.2, "courtier": -0.4}).values
    logit = (-2.6
             + 4.0 * delta_t            # ÉLASTICITÉ (forte, monotone)
             + 0.0008 * d_eur           # effet niveau absolu
             + 0.5 * (bonus_malus - 0.85)   # risque -> churn (chemin confondeur)
             - 0.06 * anciennete        # ancienneté réduit le churn
             + canal_eff
             + 0.15 * franch            # marche au passage de centaine
             - 0.10 * nb_jeunes         # captivité jeune conducteur
             + 0.6 * menace
             + rng.normal(0, 0.3, n))
    p = 1 / (1 + np.exp(-logit))
    churn = (rng.random(n) < p).astype(int)

    return pd.DataFrame({
        TARGET: churn, YEAR_COL: annee,
        "prm_propose_t": p_t, "prm_ref_t1": p_t1, "prm_t2": p_t2, "prm_t3": p_t3,
        "delta_prm_pct_t": delta_t, "delta_prm_eur_t": d_eur,
        "delta_prm_pct_t1": delta_t1, "delta_prm_pct_t2": delta_t2,
        "hausse_cumul_pct_3a": cumul, "is_franchit_centaine_t": franch,
        "anciennete": anciennete, "age_assure": age_assure, "canal": canal,
        "mode_paiement": mode_paiement, "multi_detention": multi_det,
        "classe_vehicule": classe, "zone_geo": zone, "age_vehicule": age_veh,
        "prix_vehicule": prix_veh, "nb_enfants_en_charge": nb_enf,
        "nb_jeunes_conducteurs": nb_jeunes, "bonus_malus": bonus_malus,
        "sinistralite_3a": sinistres, "nb_contacts_sc_12m": nb_contacts,
        "nb_reclamations": nb_reclam, "menace_resil_passee": menace,
        "flag_hist_incomplet": flag_inc,
    })


if USE_SYNTHETIC:
    df = make_synthetic()
else:
    # df = pd.read_parquet("ma_base.parquet")
    # df = pd.read_csv("ma_base.csv")
    raise NotImplementedError("Chargez votre base ici puis adaptez CFG.")

print(f"Base : {df.shape[0]:,} lignes x {df.shape[1]} colonnes")
df.head()

In [ ]:
# Résolution des colonnes réellement présentes (le notebook s'adapte aux trous)
def present(cols):
    return [c for c in cols if c in df.columns]

PRICE_NUM  = present([CFG[k] for k in
                      ("p_t","p_t1","p_t2","p_t3","delta_pct_t","delta_eur_t",
                       "delta_pct_t1","delta_pct_t2","cumul_3a")])
PRICE_FLAG = present([CFG["franchit_100"]])
MOD_NUM, MOD_CAT = present(MODULATORS_NUM), present(MODULATORS_CAT)
CONF_NUM, CONF_CAT = present(CONFOUNDERS_NUM), present(CONFOUNDERS_CAT)
BEH = present(BEHAVIOR)
FLG = present(FLAGS)

NUMERIC = sorted(set(PRICE_NUM + MOD_NUM + CONF_NUM + BEH))
CATEG   = sorted(set(MOD_CAT + CONF_CAT))
BINARY  = sorted(set(PRICE_FLAG + FLG + [c for c in BEH if df[c].nunique() == 2]))
NUMERIC = [c for c in NUMERIC if c not in BINARY]   # éviter doublons numérique/binaire

print("Numériques :", NUMERIC)
print("Catégorielles :", CATEG)
print("Binaires :", BINARY)
assert TARGET in df.columns, 'Colonne cible introuvable - verifiez TARGET.'

## 2. Vue d'ensemble & qualité des données

On vérifie d'abord les fondations : volumétrie, doublons, types, **taux de
résiliation** (souvent déséquilibré : 5–15 %), manquants et cardinalité.

In [ ]:
print("Dimensions :", df.shape)
print("Doublons stricts :", df.duplicated().sum())
print("Mémoire : %.1f Mo" % (df.memory_usage(deep=True).sum() / 1e6))
print("\n--- Types ---")
print(df.dtypes.value_counts())

rate = df[TARGET].mean()
print(f"\nTaux de résiliation : {rate:.2%}  (ratio {(1-rate)/rate:.1f}:1)")
df[TARGET].value_counts().plot(kind="bar", title="Distribution de la cible")
plt.xticks([0, 1], ["Conservé (0)", "Résilié (1)"], rotation=0); plt.ylabel("n"); plt.show()

In [ ]:
def missing_report(d):
    m = d.isna().sum()
    out = pd.DataFrame({"n_missing": m, "pct_missing": (m / len(d) * 100).round(2)})
    return out[out.n_missing > 0].sort_values("pct_missing", ascending=False)

miss = missing_report(df)
print("Colonnes avec valeurs manquantes :")
display(miss if len(miss) else "Aucune valeur manquante")

if len(miss):
    plt.figure(figsize=(9, 0.4 * len(miss) + 1))
    sns.barplot(x=miss.pct_missing, y=miss.index, color="#c0392b")
    plt.xlabel("% manquant"); plt.title("Valeurs manquantes par colonne"); plt.show()

In [ ]:
# Cardinalité + colonnes quasi-constantes (candidates à la suppression)
card = pd.DataFrame({
    "n_unique": df.nunique(),
    "pct_unique": (df.nunique() / len(df) * 100).round(2),
    "top_freq_pct": [df[c].value_counts(normalize=True, dropna=False).iloc[0] * 100
                     if df[c].notna().any() else np.nan for c in df.columns],
}).sort_values("n_unique")
print("Cardinalité (extrait) :")
display(card)

quasi_const = card[card.top_freq_pct > 99].index.tolist()
print("Colonnes quasi-constantes (>99% une modalité) :", quasi_const or "aucune")

## 3. Statistiques univariées (toutes les mesures)

Pour chaque variable numérique : centralité (moyenne, médiane), dispersion
(écart-type, IQR, **coefficient de variation**), forme (**skewness, kurtosis**),
quantiles étendus, **% de zéros**, et un **test de normalité** (Jarque–Bera).
Ces mesures guident transformations, capping et détection d'anomalies.

In [ ]:
def numeric_summary(d, cols):
    rows = []
    for c in cols:
        s = d[c].dropna()
        if s.empty:
            continue
        q = s.quantile([.01, .05, .25, .5, .75, .95, .99])
        jb_p = stats.jarque_bera(s)[1] if len(s) > 7 else np.nan
        rows.append({
            "n": s.size, "mean": s.mean(), "std": s.std(),
            "cv": s.std() / s.mean() if s.mean() else np.nan,
            "min": s.min(), "p01": q[.01], "p05": q[.05], "q25": q[.25],
            "median": q[.5], "q75": q[.75], "p95": q[.95], "p99": q[.99],
            "max": s.max(), "IQR": q[.75] - q[.25],
            "skew": s.skew(), "kurtosis": s.kurtosis(),
            "pct_zero": (s == 0).mean() * 100,
            "jarque_bera_p": jb_p,
        })
    return pd.DataFrame(rows, index=[c for c in cols if not d[c].dropna().empty])

stat_num = numeric_summary(df, NUMERIC)
print("Statistiques descriptives complètes :")
display(stat_num.round(3))

# Lecture rapide : variables très asymétriques (|skew|>1) -> transformation à envisager
print("Variables fortement asymétriques (|skew|>1) :",
      stat_num.index[stat_num["skew"].abs() > 1].tolist())

In [ ]:
# Histogrammes + boxplots des numériques
def grid_plots(cols, kind, ncol=3):
    if not cols:
        return
    nrow = int(np.ceil(len(cols) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(5 * ncol, 3.2 * nrow))
    axes = np.atleast_1d(axes).ravel()
    for ax, c in zip(axes, cols):
        s = df[c].dropna()
        if kind == "hist":
            sns.histplot(s, kde=True, ax=ax, color="#2980b9")
        else:
            sns.boxplot(x=s, ax=ax, color="#16a085")
        ax.set_title(c, fontsize=10); ax.set_xlabel("")
    for ax in axes[len(cols):]:
        ax.set_visible(False)
    plt.tight_layout(); plt.show()

grid_plots(NUMERIC, "hist")
grid_plots(NUMERIC, "box")

In [ ]:
# Détection d'outliers : règle IQR (1.5x) + z-score |3|
def outlier_report(d, cols):
    rows = []
    for c in cols:
        s = d[c].dropna()
        if s.empty:
            continue
        q1, q3 = s.quantile([.25, .75]); iqr = q3 - q1
        lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        iqr_out = ((s < lo) | (s > hi)).mean() * 100
        z_out = (np.abs((s - s.mean()) / s.std()) > 3).mean() * 100 if s.std() else 0
        rows.append({"pct_outliers_IQR": round(iqr_out, 2),
                     "pct_outliers_z3": round(z_out, 2),
                     "borne_basse_IQR": round(lo, 2), "borne_haute_IQR": round(hi, 2)})
    return pd.DataFrame(rows, index=[c for c in cols if not d[c].dropna().empty])

print("Outliers par variable :")
display(outlier_report(df, NUMERIC).sort_values("pct_outliers_IQR", ascending=False))

In [ ]:
# Catégorielles : fréquences + taux de résiliation par modalité
for c in CATEG:
    vc = df[c].value_counts(dropna=False)
    rate_by = df.groupby(c)[TARGET].mean().sort_values(ascending=False)
    fig, ax = plt.subplots(1, 2, figsize=(13, 3.4))
    vc.head(20).plot(kind="bar", ax=ax[0], color="#8e44ad", title=f"{c} — effectifs")
    ax[0].tick_params(axis="x", rotation=45)
    rate_by.head(20).plot(kind="bar", ax=ax[1], color="#c0392b",
                          title=f"{c} — taux de résiliation")
    ax[1].axhline(df[TARGET].mean(), ls="--", color="k", lw=1, label="taux global")
    ax[1].tick_params(axis="x", rotation=45); ax[1].legend()
    plt.tight_layout(); plt.show()

## 4. Cohérence des features prix (sanity mécanique)

**Spécifique à ce projet.** Les features prix sont dérivées et certaines seront
recalculées à l'inférence. On vérifie qu'elles sont **mécaniquement correctes** —
toute incohérence ici est un bug de construction qui fausserait l'élasticité.

In [ ]:
checks = {}

# (a) Reconstruction : P_t doit valoir P_t1 * (1 + delta_pct_t)
pt, pt1, dpt = CFG["p_t"], CFG["p_t1"], CFG["delta_pct_t"]
if all(c in df.columns for c in (pt, pt1, dpt)):
    recon = pt1_ = df[pt1] * (1 + df[dpt])
    err = (df[pt] - recon).abs() / df[pt].abs()
    checks["reconstruction_P_t (|err|<0.1%)"] = f"{(err < 1e-3).mean():.2%} cohérent"

# (b) delta_eur cohérent avec P_t - P_t1
de = CFG["delta_eur_t"]
if all(c in df.columns for c in (pt, pt1, de)):
    err2 = (df[de] - (df[pt] - df[pt1])).abs()
    checks["delta_eur = P_t - P_t1"] = f"{(err2 < 0.5).mean():.2%} cohérent"

# (c) flag_hist_incomplet cohérent avec l'ancienneté
if "flag_hist_incomplet" in df.columns and "anciennete" in df.columns:
    incoherent = ((df["flag_hist_incomplet"] == 1) & (df["anciennete"] >= 3)).sum()
    checks["flag_incomplet vs anciennete>=3"] = f"{incoherent} cas incohérents"

# (d) franchissement de centaine recalculé
fr = CFG["franchit_100"]
if all(c in df.columns for c in (fr, pt, pt1)):
    recomputed = (np.floor(df[pt] / 100) > np.floor(df[pt1] / 100)).astype(int)
    checks["is_franchit_centaine exact"] = f"{(df[fr] == recomputed).mean():.2%} cohérent"

# (e) bornes des deltas (valeurs absurdes ?)
if dpt in df.columns:
    checks[f"{dpt} min/max"] = f"[{df[dpt].min():.1%} ; {df[dpt].max():.1%}]"
    checks[f"{dpt} |delta|>50%"] = f"{(df[dpt].abs() > 0.5).mean():.2%} des lignes"

print("=== Contrôles de cohérence des features prix ===")
for k, v in checks.items():
    print(f"  • {k:38s}: {v}")

In [ ]:
# Distribution de la majoration courante : c'est le DRIVER à inspecter de près
if dpt in df.columns:
    fig, ax = plt.subplots(1, 2, figsize=(13, 3.6))
    sns.histplot(df[dpt], bins=60, kde=True, ax=ax[0], color="#2980b9")
    ax[0].axvline(0, color="k", ls="--"); ax[0].set_title(f"{dpt} — distribution")
    sns.boxplot(x=df[dpt], ax=ax[1], color="#2980b9")
    ax[1].set_title(f"{dpt} — boxplot")
    plt.tight_layout(); plt.show()
    print(df[dpt].describe(percentiles=[.01, .05, .5, .95, .99]).round(4))

## 5. Cœur élasticité : courbes prix → résiliation

**La section qui décide si le projet est viable.** On bucketise la majoration et
on trace le **taux de résiliation observé par tranche**, avec intervalles de
confiance de **Wilson**. On attend une **pente croissante et ~monotone**.

- Plate → pas d'élasticité exploitable, ou noyée par un confondeur.
- Non-monotone → alerte qualité de données (l'optimiseur exploiterait les creux).

On segmente ensuite (ancienneté, canal…) pour voir si **l'élasticité varie par
segment** — ce qui justifie les interactions prix × segment dans le modèle.

In [ ]:
def wilson_ci(k, n, z=1.96):
    if n == 0:
        return (np.nan, np.nan)
    p = k / n; d = 1 + z**2 / n
    c = (p + z**2 / (2 * n)) / d
    h = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / d
    return c - h, c + h

def churn_curve(d, feat, q=10, ax=None, label=None, color=None):
    s = d[[feat, TARGET]].dropna().copy()
    try:
        s["bk"] = pd.qcut(s[feat], q, duplicates="drop")
    except ValueError:
        s["bk"] = pd.cut(s[feat], min(q, s[feat].nunique()))
    g = s.groupby("bk")[TARGET].agg(["mean", "count", "sum"])
    ci = [wilson_ci(r["sum"], r["count"]) for _, r in g.iterrows()]
    g["lo"], g["hi"] = [c[0] for c in ci], [c[1] for c in ci]
    g["x"] = [iv.mid for iv in g.index]
    ax = ax or plt.gca()
    ax.plot(g["x"], g["mean"], marker="o", label=label, color=color)
    ax.fill_between(g["x"], g["lo"], g["hi"], alpha=0.15, color=color)
    # test de monotonie : Spearman entre rang du bucket et taux
    rho, pval = stats.spearmanr(range(len(g)), g["mean"])
    return g, rho, pval

plt.figure(figsize=(9, 5))
g, rho, pval = churn_curve(df, dpt, q=12, color="#c0392b", label="observé")
plt.axhline(df[TARGET].mean(), ls="--", color="k", lw=1, label="taux global")
plt.title(f"Élasticité brute : résiliation vs {dpt}")
plt.xlabel(f"{dpt} (centre de tranche)"); plt.ylabel("taux de résiliation")
plt.legend(); plt.show()
print(f"Monotonie (Spearman rang vs taux) : rho={rho:.3f}  p={pval:.1e}")
print("=> rho proche de +1 = élasticité propre et croissante." )
display(g[["mean", "count", "lo", "hi"]].round(4))

In [ ]:
# Élasticité segmentée : la pente doit varier selon le segment
def segmented_curve(feat, seg, q=8, top=4):
    plt.figure(figsize=(9, 5))
    if df[seg].dtype.kind in "biufc" and df[seg].nunique() > top:
        df["_seg"] = pd.qcut(df[seg], top, duplicates="drop").astype(str)
        groups = sorted(df["_seg"].dropna().unique())
        getter = lambda gname: df[df["_seg"] == gname]
    else:
        groups = df[seg].value_counts().head(top).index.tolist()
        getter = lambda gname: df[df[seg] == gname]
    cols = sns.color_palette("viridis", len(groups))
    for gname, col in zip(groups, cols):
        sub = getter(gname)
        if len(sub) > 200:
            churn_curve(sub, feat, q=q, color=col, label=str(gname))
    plt.title(f"Élasticité ({feat}) segmentée par {seg}")
    plt.xlabel(feat); plt.ylabel("taux de résiliation"); plt.legend(title=seg)
    plt.show()
    df.drop(columns=[c for c in ["_seg"] if c in df.columns], inplace=True)

for seg in [c for c in ["anciennete", "canal", "prix_vehicule"] if c in df.columns]:
    segmented_curve(dpt, seg)

In [ ]:
# Effet de SEUIL psychologique : résiliation juste sous vs juste au-dessus d'une centaine
if CFG["p_t"] in df.columns:
    p = df[CFG["p_t"]]
    dist = (p % 100)                      # 0 = pile sur une centaine
    pos = np.where(dist <= 10, "0-10€ au-dessus seuil",
          np.where(dist >= 90, "0-10€ sous seuil", "milieu de tranche"))
    tmp = df.assign(_pos=pos)
    seuil = tmp.groupby("_pos")[TARGET].agg(["mean", "count"])
    print("Taux de résiliation autour des centaines :")
    display(seuil.round(4))
    print("=> marche nette sous/au-dessus => garder is_franchit_centaine ; "
          "sinon c'est du bruit que l'optimiseur sur-exploiterait.")

## 6. Confounding prix ↔ risque (le piège n°1)

Si le prix a été historiquement fixé selon le risque, le prix **corrèle** avec le
risque et le modèle attribuera au prix un effet qui est en réalité du risque.
On mesure cette corrélation, puis on retrace l'élasticité **à risque constant**
(au sein de strates de risque) : si la pente survit, l'élasticité est réelle ;
si elle s'effondre, elle était du risque déguisé.

In [ ]:
# Corrélation des features prix avec les variables de risque
risk_cols = [c for c in CONF_NUM if c in df.columns]
price_cols = [c for c in [CFG["p_t1"], CFG["delta_pct_t"], CFG["delta_eur_t"]]
              if c in df.columns]
if risk_cols and price_cols:
    cmat = df[price_cols + risk_cols].corr(method="spearman").loc[price_cols, risk_cols]
    plt.figure(figsize=(1.8 * len(risk_cols) + 2, 1.0 * len(price_cols) + 1.5))
    sns.heatmap(cmat, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
                vmin=-1, vmax=1, cbar_kws={"label": "Spearman"})
    plt.title("Corrélation prix ↔ risque (confounding)"); plt.tight_layout(); plt.show()
    print("Lecture : forte corrélation sur le NIVEAU (P_t1) = confounding attendu.")
    print("La majoration (delta) est en général plus propre que le niveau.")

In [ ]:
# Élasticité À RISQUE CONSTANT : courbe delta->churn dans chaque strate de risque
if RISK_STRATIFY in df.columns:
    df["_risk_bin"] = pd.qcut(df[RISK_STRATIFY], 4, duplicates="drop")
    plt.figure(figsize=(9, 5))
    cols = sns.color_palette("rocket", df["_risk_bin"].nunique())
    for (rb, col) in zip(sorted(df["_risk_bin"].dropna().unique()), cols):
        sub = df[df["_risk_bin"] == rb]
        if len(sub) > 200:
            churn_curve(sub, dpt, q=6, color=col, label=f"{RISK_STRATIFY} {rb}")
    plt.title(f"Élasticité de {dpt} à risque constant (strates de {RISK_STRATIFY})")
    plt.xlabel(dpt); plt.ylabel("taux de résiliation"); plt.legend(fontsize=8)
    plt.show()
    df.drop(columns=["_risk_bin"], inplace=True)
    print("=> Si la pente persiste dans chaque strate : élasticité réelle, pas un artefact de risque.")

## 7. Corrélations, redondances (VIF) & associations

On chasse la **redondance** (essentiel vu les variables véhicule/risque très
colinéaires) avec corrélations Pearson/Spearman, **VIF** (multicolinéarité),
**Cramér's V** (catégorielle ↔ catégorielle), **rapport de corrélation η**
(catégorielle ↔ numérique) et **information mutuelle** vs la cible.

In [ ]:
# Matrice de corrélation Spearman (robuste aux non-linéarités) + paires redondantes
corr = df[NUMERIC].corr(method="spearman")
plt.figure(figsize=(0.55 * len(NUMERIC) + 3, 0.55 * len(NUMERIC) + 2))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=False, cmap="RdBu_r", center=0, vmin=-1, vmax=1,
            square=True, cbar_kws={"label": "Spearman"})
plt.title("Corrélations numériques (Spearman)"); plt.tight_layout(); plt.show()

pairs = (corr.where(~mask).stack()
         .rename("corr").reset_index()
         .query("abs(corr) > 0.7")
         .sort_values("corr", key=abs, ascending=False))
print("Paires fortement corrélées (|rho|>0.7) — candidates à élaguer :")
display(pairs)

In [ ]:
# VIF (multicolinéarité) calculé via R² des régressions linéaires croisées
def vif_table(d, cols):
    X = d[cols].dropna()
    if X.shape[0] < 50 or X.shape[1] < 2:
        return pd.Series(dtype=float)
    Xv = X.values.astype(float); out = {}
    for i, c in enumerate(cols):
        y = Xv[:, i]
        A = np.column_stack([np.ones(len(Xv)), np.delete(Xv, i, axis=1)])
        beta, *_ = np.linalg.lstsq(A, y, rcond=None)
        ss_res = ((y - A @ beta) ** 2).sum()
        ss_tot = ((y - y.mean()) ** 2).sum()
        r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0
        out[c] = np.inf if r2 >= 1 else 1 / (1 - r2)
    return pd.Series(out).sort_values(ascending=False)

vif = vif_table(df, NUMERIC)
print("VIF (>5 = colinéarité notable, >10 = forte) :")
display(vif.round(2).to_frame("VIF"))

In [ ]:
# Associations catégorielles : Cramér's V (corrigé)
def cramers_v(x, y):
    ct = pd.crosstab(x, y)
    if ct.shape[0] < 2 or ct.shape[1] < 2:
        return np.nan
    chi2 = stats.chi2_contingency(ct)[0]; n = ct.values.sum()
    r, k = ct.shape; phi2 = chi2 / n
    phi2c = max(0, phi2 - (k - 1) * (r - 1) / (n - 1))
    rc = r - (r - 1) ** 2 / (n - 1); kc = k - (k - 1) ** 2 / (n - 1)
    denom = max(min(kc - 1, rc - 1), 1e-9)
    return np.sqrt(phi2c / denom)

if len(CATEG) >= 2:
    cv = pd.DataFrame(index=CATEG, columns=CATEG, dtype=float)
    for a in CATEG:
        for b in CATEG:
            cv.loc[a, b] = 1.0 if a == b else cramers_v(df[a], df[b])
    plt.figure(figsize=(0.7 * len(CATEG) + 2, 0.7 * len(CATEG) + 1))
    sns.heatmap(cv.astype(float), annot=True, fmt=".2f", cmap="Purples", vmin=0, vmax=1)
    plt.title("Cramér's V (catégorielles)"); plt.tight_layout(); plt.show()

In [ ]:
# Rapport de corrélation η : force de lien catégorielle -> numérique (ex: classe -> prix)
def correlation_ratio(cats, values):
    s = pd.DataFrame({"c": cats, "v": values}).dropna()
    if s.empty:
        return np.nan
    grand = s["v"].mean()
    ss_between = s.groupby("c")["v"].apply(lambda g: len(g) * (g.mean() - grand) ** 2).sum()
    ss_total = ((s["v"] - grand) ** 2).sum()
    return np.sqrt(ss_between / ss_total) if ss_total > 0 else 0

if CATEG and NUMERIC:
    eta = pd.DataFrame(index=CATEG, columns=NUMERIC, dtype=float)
    for c in CATEG:
        for n_ in NUMERIC:
            eta.loc[c, n_] = correlation_ratio(df[c], df[n_])
    plt.figure(figsize=(0.5 * len(NUMERIC) + 3, 0.5 * len(CATEG) + 2))
    sns.heatmap(eta.astype(float), annot=True, fmt=".2f", cmap="Greens", vmin=0, vmax=1)
    plt.title("Rapport de corrélation η (catégorielle → numérique)")
    plt.tight_layout(); plt.show()

In [ ]:
# Information mutuelle vs la cible : pouvoir prédictif brut (capte le non-linéaire)
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import OrdinalEncoder

feat_cols = list(dict.fromkeys(NUMERIC + CATEG + BINARY))
cat_like = list(dict.fromkeys(CATEG + BINARY))
X = df[feat_cols].copy()
for c in cat_like:
    X[c] = OrdinalEncoder(handle_unknown="use_encoded_value",
                          unknown_value=-1).fit_transform(
                          X[[c]].astype(str)).ravel()
X = X.fillna(X.median(numeric_only=True)).fillna(-1)
disc = [X.columns.get_loc(c) for c in cat_like]
mi = mutual_info_classif(X, df[TARGET], discrete_features=disc, random_state=0)
mi_s = pd.Series(mi, index=feat_cols).sort_values(ascending=False)
plt.figure(figsize=(8, 0.35 * len(mi_s) + 1))
sns.barplot(x=mi_s.values, y=mi_s.index, color="#2c3e50")
plt.title("Information mutuelle avec la résiliation"); plt.xlabel("MI"); plt.show()
display(mi_s.round(4).to_frame("mutual_info"))

## 8. Stabilité temporelle (PSI)

Une rupture entre millésimes (changement de politique tarifaire, choc
concurrentiel) polluerait un split aléatoire et impose une **validation
out-of-time**. On suit le taux de résiliation et la distribution de la majoration
par année, et on calcule le **PSI** (Population Stability Index) du driver.

In [ ]:
if YEAR_COL and YEAR_COL in df.columns:
    by_year = df.groupby(YEAR_COL).agg(
        n=(TARGET, "size"), taux_resil=(TARGET, "mean"))
    if dpt in df.columns:
        by_year["majoration_med"] = df.groupby(YEAR_COL)[dpt].median()
    print("Évolution par millésime :"); display(by_year.round(4))

    fig, ax = plt.subplots(1, 2, figsize=(13, 3.6))
    by_year["taux_resil"].plot(marker="o", ax=ax[0], title="Taux de résiliation / an")
    ax[0].axhline(df[TARGET].mean(), ls="--", color="k", lw=1)
    if dpt in df.columns:
        sns.boxplot(data=df, x=YEAR_COL, y=dpt, ax=ax[1])
        ax[1].set_title(f"{dpt} par millésime")
    plt.tight_layout(); plt.show()
else:
    print("Pas de colonne millésime (YEAR_COL) — section ignorée.")

In [ ]:
# PSI du driver entre la 1re et la dernière année (>0.25 = dérive majeure)
def psi(expected, actual, bins=10):
    e, a = expected.dropna(), actual.dropna()
    if len(e) < 50 or len(a) < 50:
        return np.nan
    cuts = np.quantile(e, np.linspace(0, 1, bins + 1)); cuts[0], cuts[-1] = -np.inf, np.inf
    e_pct = np.histogram(e, cuts)[0] / len(e)
    a_pct = np.histogram(a, cuts)[0] / len(a)
    e_pct, a_pct = np.clip(e_pct, 1e-4, None), np.clip(a_pct, 1e-4, None)
    return float(np.sum((a_pct - e_pct) * np.log(a_pct / e_pct)))

if YEAR_COL in df.columns and dpt in df.columns:
    yrs = sorted(df[YEAR_COL].dropna().unique())
    if len(yrs) >= 2:
        base = df[df[YEAR_COL] == yrs[0]]
        print(f"PSI de {dpt} (référence = {yrs[0]}) :")
        for y in yrs[1:]:
            val = psi(base[dpt], df[df[YEAR_COL] == y][dpt])
            tag = "OK" if val < 0.1 else ("à surveiller" if val < 0.25 else "DÉRIVE")
            print(f"  {yrs[0]} -> {y} : PSI = {val:.3f}  [{tag}]")

## 9. Scan de fuite (leakage)

Pour chaque feature : **était-elle connue au moment de l'offre ?** Une corrélation
ou information mutuelle anormalement forte avec la cible est un signal de fuite
avant d'être un bon prédicteur (ex. prime réellement payée, comportement
post-renouvellement). On liste les suspects à revoir **manuellement**.

In [ ]:
# Corrélation point-bisériale (numérique <-> cible binaire) + repérage des suspects
pb = {}
for c in NUMERIC + BINARY:
    s = df[[c, TARGET]].dropna()
    if s[c].nunique() > 1:
        pb[c] = stats.pointbiserialr(s[TARGET], s[c])[0]
pb_s = pd.Series(pb).sort_values(key=abs, ascending=False)

suspects = pd.DataFrame({"point_biserial": pb_s})
suspects["mutual_info"] = mi_s.reindex(suspects.index)
suspects["ALERTE_leakage"] = (suspects["point_biserial"].abs() > 0.5) | (suspects["mutual_info"] > 0.2)
print("Top liens avec la cible (vérifier les ALERTE = fuite potentielle) :")
display(suspects.head(15).round(4))
flagged = suspects.index[suspects["ALERTE_leakage"]].tolist()
print("\n⚠️ À auditer manuellement (connu au moment de l'offre ?) :", flagged or "aucun")

## 10. Synthèse automatique

Récapitulatif des signaux à traiter avant modélisation. À confronter à votre
connaissance métier — ces règles sont des garde-fous, pas des verdicts.

In [ ]:
print("=" * 64)
print(" SYNTHÈSE EDA — MODÈLE DE RÉSILIATION TARIFAIRE")
print("=" * 64)

print(f"\n[Base]   {df.shape[0]:,} lignes, {df.shape[1]} colonnes | "
      f"taux résiliation = {df[TARGET].mean():.2%}")

if len(miss):
    print(f"\n[Manquants] {len(miss)} colonne(s) :")
    for c, r in miss.pct_missing.items():
        print(f"   - {c}: {r:.1f}%  (NaN natif + flag, JAMAIS imputer le prix à 0)")
else:
    print("\n[Manquants] aucun")

print(f"\n[Élasticité brute] monotonie {dpt} : rho={rho:.3f}")
print("   -> rho>0.8 = signal exploitable ; faible/négatif = creuser le confounding")

if 'pairs' in dir() and len(pairs):
    print(f"\n[Redondance] {len(pairs)} paire(s) |rho|>0.7 — élaguer (cf. variables véhicule)")
if (vif > 10).any():
    print(f"[VIF>10] colinéarité forte : {vif.index[vif > 10].tolist()}")

if quasi_const:
    print(f"\n[Quasi-constantes] à supprimer : {quasi_const}")

if flagged:
    print(f"\n[⚠ Leakage potentiel] auditer : {flagged}")

print("\n[Rappels modélisation]")
print("   • Monotonie forcée sur le bloc prix (delta_pct_t, delta_eur_t, cumul_3a)")
print("   • Interactions prix×segment : natives sur arbres, À ACTIVER sur EBM (GA2M)")
print("   • Validation OUT-OF-TIME (jamais split aléatoire) + calibration des probas")
print("   • Borner le balayage de l'optimiseur à la plage de prix observée")
print("=" * 64)